# 02 阻尼 Newton、弦 Newton 与有限差分 Jacobian

上一节的 Newton 法每步使用当前 Jacobian $J_F(x_k)$ 并取完整 Newton 步。实际问题中这一步可能太激进，也可能太昂贵：

* 远离根时，完整 Newton 步可能增大残差；
* 解析 Jacobian 可能难以推导或实现；
* 每步重新分解 Jacobian 的成本可能过高。

本节介绍三个直接改造：阻尼 Newton、有限差分 Jacobian 和弦 Newton。


In [ ]:
import math
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
while ROOT.name != "py-sc" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import chord_newton_system_method, damped_newton_system_method, finite_difference_jacobian, newton_system_method


## 有限差分 Jacobian

若解析 Jacobian 不方便，可以按列做前向差分：

$$
J_F(x)e_j \approx \frac{F(x+h_j e_j)-F(x)}{h_j}.
$$

步长过大会引入截断误差，步长过小会放大舍入误差。教学实现使用与分量尺度相关的默认步长。


In [ ]:
def F(x):
    return np.array([x[0] ** 2 + x[1] ** 2 - 1.0, x[0] - x[1]])


def J(x):
    return np.array([[2.0 * x[0], 2.0 * x[1]], [1.0, -1.0]])

point = np.array([0.8, 0.6])
fd_jacobian = finite_difference_jacobian(F, point)
analytic = J(point)

print("finite-difference Jacobian:")
print(np.array2string(fd_jacobian, precision=8))
print("analytic Jacobian:")
print(np.array2string(analytic, precision=8))
print("error norm:", f"{np.linalg.norm(fd_jacobian - analytic):.3e}")


## 阻尼 Newton

阻尼 Newton 将完整步 $s_k$ 改为

$$
x_{k+1}=x_k+\lambda_k s_k,
\qquad 0<\lambda_k\le 1,
$$

并通过回溯选择能降低残差范数的 $\lambda_k$。这会牺牲局部速度，但能改善远离根时的稳定性。


In [ ]:
def cubic_system(x):
    return np.array([x[0] ** 3 - 1.0, x[1]])


def cubic_jacobian(x):
    return np.array([[3.0 * x[0] ** 2, 0.0], [0.0, 1.0]])

first_full_step = np.linalg.solve(cubic_jacobian(np.array([0.1, 0.5])), -cubic_system(np.array([0.1, 0.5])))
damped = damped_newton_system_method(cubic_system, cubic_jacobian, [0.1, 0.5], tolerance=1e-12)

print("undamped first trial:", np.array2string(np.array([0.1, 0.5]) + first_full_step, precision=8))
print("damped solution:", np.array2string(damped.solution, precision=12))
print("iterations:", damped.iterations, "residual:", f"{damped.residual_norm:.3e}")
print("residual history:", np.array2string(damped.residual_history, precision=3))


## 弦 Newton

弦 Newton 固定某个 Jacobian 近似 $B$，反复求解

$$
B s_k=-F(x_k).
$$

如果 $B$ 是根附近的好近似，方法可以收敛；如果 $B$ 很差，就会变慢甚至失败。它的优势是只需分解一次线性系统矩阵，适合 Jacobian 构造或分解很贵但变化较慢的场景。


In [ ]:
full_newton = newton_system_method(F, J, [0.8, 0.6], tolerance=1e-12)
chord = chord_newton_system_method(F, J, [0.8, 0.6], tolerance=1e-10, max_iterations=40)
expected = np.array([1.0 / math.sqrt(2.0), 1.0 / math.sqrt(2.0)])

print("full Newton iterations:", full_newton.iterations)
print("chord Newton iterations:", chord.iterations)
print("chord solution:", np.array2string(chord.solution, precision=12))
print("expected:", np.array2string(expected, precision=12))


## 本节小结

阻尼 Newton、有限差分 Jacobian 和弦 Newton 都是在“速度、稳健性、单步成本”之间做权衡。阻尼回溯提高远离根时的可靠性；有限差分 Jacobian 降低推导成本，但引入步长误差；弦 Newton 降低重复分解成本，却可能需要更多迭代。这些思想为下一节的 Broyden 拟 Newton 方法铺垫：不再每步重算 Jacobian，而是用迭代信息更新近似矩阵。
